# Business Insights & Operational Recommendations

This notebook synthesizes findings from the EDA, feature engineering, driver analysis, and predictive model into **actionable business intelligence** for Uber Eats operations.

## Audience
Operations managers, dispatch planners, courier incentive teams, and product stakeholders.

## Key Business Questions Answered
1. **Where** are the biggest ATD inefficiencies (territory, time, courier type)?
2. **What** drives late deliveries — distance, demand peaks, or courier behavior?
3. **Who** are the high-impact drivers to prioritize for coaching or incentives?
4. **How** can we reduce mean ATD and improve ETA reliability?
5. **What** should operations do differently next week?

---
**Sections**
1. Setup & Data Load
2. Executive KPI Dashboard
3. ATD Efficiency Heatmap by Segment
4. Demand × Supply Mismatch Analysis
5. Distance-Based Opportunity Sizing
6. Courier Flow Optimization Opportunities
7. Driver Incentive Targeting
8. Model Business Value Quantification
9. Strategic Recommendations
10. Execution Roadmap

---
## 1. Setup & Data Load

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 5)})
SEED = 42

RAW_PATH      = Path('../data/raw/BC_A&A_with_ATD.csv')
FEATURES_PATH = Path('../data/processed/features.parquet')

In [ ]:
if FEATURES_PATH.exists():
    df = pd.read_parquet(FEATURES_PATH)
    print(f'Loaded processed features: {df.shape}')
else:
    dtype_map = {
        'region': 'category', 'territory': 'category',
        'country_name': 'category', 'courier_flow': 'category',
        'geo_archetype': 'category', 'merchant_surface': 'category',
    }
    df = pd.read_csv(
        RAW_PATH, dtype=dtype_map,
        parse_dates=['restaurant_offered_timestamp_utc',
                     'order_final_state_timestamp_local',
                     'eater_request_timestamp_local'],
        na_values=['\\N', 'NULL', 'None', ''],
    )
    df = df[df['ATD'] > 0].copy()
    df['ATD_capped'] = df['ATD'].clip(upper=df['ATD'].quantile(0.99))
    ts = df['restaurant_offered_timestamp_utc']
    df['hour_local']   = (ts.dt.hour - 6) % 24
    df['day_of_week']  = ts.dt.dayofweek
    df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_peak_hour'] = df['hour_local'].isin(list(range(12, 15)) + list(range(19, 24))).astype(int)

    def time_block(h):
        if 6 <= h < 11:  return 'morning'
        if 11 <= h < 14: return 'lunch'
        if 14 <= h < 18: return 'afternoon'
        if 18 <= h < 22: return 'dinner'
        if 22 <= h < 24: return 'late_night'
        return 'overnight'
    df['time_block'] = df['hour_local'].apply(time_block).astype('category')

    for c in ['pickup_distance', 'dropoff_distance']:
        df[c].fillna(df[c].median(), inplace=True)
    df['total_distance'] = df['pickup_distance'] + df['dropoff_distance']
    print(f'Built from raw: {df.shape}')

ATD_COL = 'ATD_capped' if 'ATD_capped' in df.columns else 'ATD'
DOW_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
BLOCK_ORDER = ['overnight', 'morning', 'lunch', 'afternoon', 'dinner', 'late_night']

---
## 2. Executive KPI Dashboard

In [ ]:
# Core KPIs
total_trips     = len(df)
atd_median      = df[ATD_COL].median()
atd_mean        = df[ATD_COL].mean()
atd_p95         = df[ATD_COL].quantile(0.95)
atd_p99         = df[ATD_COL].quantile(0.99)
late_rate_30    = (df[ATD_COL] > 30).mean() * 100  # % trips > 30 min (SLA example)
late_rate_45    = (df[ATD_COL] > 45).mean() * 100
late_rate_60    = (df[ATD_COL] > 60).mean() * 100
unique_drivers  = df['driver_uuid'].nunique()
unique_terr     = df['territory'].nunique()
date_range_days = (df['restaurant_offered_timestamp_utc'].max() -
                   df['restaurant_offered_timestamp_utc'].min()).days
daily_avg_trips = total_trips / (date_range_days or 1)

kpis = {
    '📦 Total Trips':                   f'{total_trips:,}',
    '📅 Period (days)':                  f'{date_range_days}',
    '🚀 Avg Daily Trips':               f'{daily_avg_trips:,.0f}',
    '🕐 Median ATD':                     f'{atd_median:.1f} min',
    '📊 Mean ATD':                       f'{atd_mean:.1f} min',
    '📈 p95 ATD':                        f'{atd_p95:.1f} min',
    '🔴 % Trips > 30 min (SLA breach)': f'{late_rate_30:.1f}%',
    '🔴 % Trips > 45 min':              f'{late_rate_45:.1f}%',
    '🔴 % Trips > 60 min':              f'{late_rate_60:.1f}%',
    '🚗 Unique Drivers':                f'{unique_drivers:,}',
    '🗺️  Unique Territories':            f'{unique_terr}',
}

print('╔' + '═' * 55 + '╗')
print('║        UBER EATS MEXICO — DELIVERY PERFORMANCE KPIs      ║')
print('╠' + '═' * 55 + '╣')
for k, v in kpis.items():
    print(f'║  {k:<38}  {v:>12}  ║')
print('╚' + '═' * 55 + '╝')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4))

# 1. ATD cumulative distribution with SLA lines
atd_sorted = np.sort(df[ATD_COL].values)
cdf = np.arange(1, len(atd_sorted) + 1) / len(atd_sorted)
axes[0].plot(atd_sorted[:int(len(atd_sorted) * 0.99)],
             cdf[:int(len(atd_sorted) * 0.99)],
             color='steelblue', linewidth=2)
for sla, col in [(30, 'orange'), (45, 'red'), (60, 'darkred')]:
    pct_below = (atd_sorted <= sla).mean() * 100
    axes[0].axvline(sla, color=col, linestyle='--', alpha=0.8,
                    label=f'{sla} min → {pct_below:.0f}% on time')
axes[0].set_xlabel('ATD (min)')
axes[0].set_ylabel('CDF')
axes[0].set_title('ATD Cumulative Distribution')
axes[0].legend(fontsize=8)

# 2. Daily trip volume trend
daily = (df.groupby(df['restaurant_offered_timestamp_utc'].dt.date)
           .size().reset_index(name='trips'))
daily['date'] = pd.to_datetime(daily['restaurant_offered_timestamp_utc'])
axes[1].fill_between(daily['date'], daily['trips'], alpha=0.3, color='teal')
axes[1].plot(daily['date'], daily['trips'], color='teal', linewidth=1.5)
axes[1].set_title('Daily Trip Volume')
axes[1].set_xlabel('Date')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# 3. ATD distribution by courier flow
flow_order = df.groupby('courier_flow', observed=True)[ATD_COL].median().sort_values().index
sample50k = df.sample(min(50_000, len(df)), random_state=SEED)
sns.boxplot(data=sample50k, x='courier_flow', y=ATD_COL, order=flow_order,
            ax=axes[2], palette='muted',
            flierprops=dict(marker='.', markersize=1, alpha=0.2))
axes[2].set_title('ATD by Courier Flow')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=20)

# 4. % on-time by time block
if 'time_block' in df.columns:
    ontime = (df.groupby('time_block', observed=True)
               .apply(lambda x: (x[ATD_COL] <= 30).mean() * 100)
               .reindex(BLOCK_ORDER))
    bar_colors = ['seagreen' if v >= 60 else 'coral' for v in ontime.values]
    ontime.plot(kind='bar', ax=axes[3], color=bar_colors, edgecolor='none')
    axes[3].axhline(60, color='black', linestyle='--', linewidth=0.8, label='60% target')
    axes[3].set_title('% On-Time (≤30 min) by Time Block')
    axes[3].set_ylabel('%')
    axes[3].tick_params(axis='x', rotation=20)
    axes[3].legend(fontsize=9)

plt.suptitle('Executive KPI Overview', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. ATD Efficiency Heatmap by Segment

Identify which territory × time combinations perform worst — these are the **top intervention targets**.

In [ ]:
if 'time_block' in df.columns:
    pivot_atd = (
        df.groupby(['territory', 'time_block'], observed=True)[ATD_COL]
        .median()
        .unstack('time_block')
        .reindex(columns=BLOCK_ORDER)
    )
    # Sort territories by worst overall median ATD
    pivot_atd = pivot_atd.loc[pivot_atd.median(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(14, max(6, len(pivot_atd) * 0.45)))
    sns.heatmap(
        pivot_atd, ax=ax, cmap='YlOrRd',
        annot=True, fmt='.0f',
        cbar_kws={'label': 'Median ATD (min)'},
        linewidths=0.3,
    )
    ax.set_title('Median ATD (min) — Territory × Time Block', fontweight='bold', fontsize=13)
    ax.set_xlabel('Time Block')
    ax.set_ylabel('Territory')
    plt.tight_layout()
    plt.show()

    # Find critical cells: ATD > global p75 AND volume > 1% of total trips
    p75_atd = df[ATD_COL].quantile(0.75)
    vol_threshold = len(df) * 0.01

    seg_vol = (
        df.groupby(['territory', 'time_block'], observed=True)
        .agg(trips=(ATD_COL, 'count'), atd_median=(ATD_COL, 'median'))
        .reset_index()
    )
    critical_segs = seg_vol[
        (seg_vol['atd_median'] > p75_atd) &
        (seg_vol['trips'] > vol_threshold)
    ].sort_values('atd_median', ascending=False)

    print(f'Critical segments (ATD > p75={p75_atd:.0f} min AND >1% of trips):')
    display(critical_segs.round(1))

---
## 4. Demand × Supply Mismatch Analysis

ATD spikes during peak demand hours suggest courier supply isn't keeping up. We quantify the mismatch.

In [ ]:
if 'hour_local' in df.columns:
    hourly = (
        df.groupby('hour_local')
        .agg(
            trip_volume=(ATD_COL, 'count'),
            atd_median=(ATD_COL, 'median'),
            atd_p75=(ATD_COL, lambda x: x.quantile(0.75)),
        )
    )

    # Normalize volume to [0,1]
    hourly['volume_norm'] = hourly['trip_volume'] / hourly['trip_volume'].max()
    hourly['atd_norm']    = hourly['atd_median'] / hourly['atd_median'].max()

    # Mismatch score: high demand + high ATD = supply gap
    hourly['demand_supply_gap'] = hourly['volume_norm'] * hourly['atd_norm']

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    x = hourly.index
    ax.bar(x, hourly['trip_volume'], color='steelblue', alpha=0.6, label='Trip Volume')
    ax2 = ax.twinx()
    ax2.plot(x, hourly['atd_median'], color='tomato', marker='o', linewidth=2, label='Median ATD')
    ax.set_xlabel('Local Hour')
    ax.set_ylabel('Trip Volume', color='steelblue')
    ax2.set_ylabel('Median ATD (min)', color='tomato')
    ax.set_title('Demand Volume vs ATD by Hour')
    ax.set_xticks(range(0, 24))
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    # Gap score
    hourly['demand_supply_gap'].plot(
        kind='bar', ax=axes[1],
        color=['tomato' if v > hourly['demand_supply_gap'].quantile(0.75) else 'steelblue'
               for v in hourly['demand_supply_gap'].values],
        edgecolor='none'
    )
    axes[1].set_title('Demand-Supply Gap Score by Hour')
    axes[1].set_xlabel('Local Hour')
    axes[1].set_ylabel('Gap Score (volume × ATD normalized)')
    axes[1].tick_params(axis='x', rotation=0)

    plt.suptitle('Demand vs Supply Analysis', fontweight='bold')
    plt.tight_layout()
    plt.show()

    peak_hours = hourly[hourly['demand_supply_gap'] > hourly['demand_supply_gap'].quantile(0.75)].index.tolist()
    print(f'High demand-supply gap hours (local): {sorted(peak_hours)}')

---
## 5. Distance-Based Opportunity Sizing

Distance is the biggest driver of ATD. We quantify how much ATD reduction is achievable through better courier dispatch (shorter pickup distance).

In [ ]:
if 'total_distance' in df.columns:
    p99_dist = df['total_distance'].quantile(0.99)
    df_dist = df[df['total_distance'] <= p99_dist].copy()

    dist_bins   = [0, 1, 2, 3, 5, 8, 15, np.inf]
    dist_labels = ['0-1', '1-2', '2-3', '3-5', '5-8', '8-15', '15+']
    df_dist['total_dist_bin'] = pd.cut(
        df_dist['total_distance'], bins=dist_bins, labels=dist_labels, include_lowest=True
    )

    dist_analysis = (
        df_dist.groupby('total_dist_bin', observed=True)
        .agg(
            trips=(ATD_COL, 'count'),
            atd_median=(ATD_COL, 'median'),
            atd_mean=(ATD_COL, 'mean'),
        )
        .assign(trip_pct=lambda x: x['trips'] / x['trips'].sum() * 100)
    )

    # Opportunity: if all trips were ≤ 3km total, what's the ATD gain?
    atd_if_short = df_dist[df_dist['total_distance'] <= 3][ATD_COL].median()
    atd_current  = df_dist[ATD_COL].median()
    pct_long     = (df_dist['total_distance'] > 3).mean() * 100

    print('ATD by total distance bucket:')
    display(dist_analysis.round(2))
    print(f'\nCurrent median ATD    : {atd_current:.1f} min')
    print(f'Median ATD (≤3km)     : {atd_if_short:.1f} min')
    print(f'Trips > 3km (long)    : {pct_long:.1f}%')
    print(f'Potential ATD saving  : {atd_current - atd_if_short:.1f} min if long trips reassigned')

In [ ]:
if 'pickup_distance' in df.columns:
    # How much of ATD variance is explained by pickup vs dropoff distance?
    corr_pickup  = df[['pickup_distance', ATD_COL]].dropna().corr().iloc[0, 1]
    corr_dropoff = df[['dropoff_distance', ATD_COL]].dropna().corr().iloc[0, 1]

    print('Distance correlation with ATD:')
    print(f'  Pickup distance  (r={corr_pickup:.3f})  — controllable via dispatch')
    print(f'  Dropoff distance (r={corr_dropoff:.3f})  — fixed by customer location')
    print()
    print('Insight: Pickup distance is actionable (assign closer couriers).')
    print('Dropoff distance is determined at order placement (ETA estimation input).')

    # Quantify: trips where pickup_distance > 1.5km (dispatch improvement target)
    long_pickup_pct = (df['pickup_distance'] > 1.5).mean() * 100
    long_pickup_atd = df[df['pickup_distance'] > 1.5][ATD_COL].median()
    short_pickup_atd = df[df['pickup_distance'] <= 1.5][ATD_COL].median()
    print(f'\nTrips with pickup > 1.5km  : {long_pickup_pct:.1f}%')
    print(f'  ATD (long pickup)        : {long_pickup_atd:.1f} min')
    print(f'  ATD (short pickup ≤1.5k) : {short_pickup_atd:.1f} min')
    print(f'  ATD gap                  : {long_pickup_atd - short_pickup_atd:.1f} min')

---
## 6. Courier Flow Optimization Opportunities

In [ ]:
flow_summary = (
    df.groupby('courier_flow', observed=True)
    .agg(
        trips=(ATD_COL, 'count'),
        atd_median=(ATD_COL, 'median'),
        atd_p75=(ATD_COL, lambda x: x.quantile(0.75)),
        late_30_pct=(ATD_COL, lambda x: (x > 30).mean() * 100),
        late_45_pct=(ATD_COL, lambda x: (x > 45).mean() * 100),
    )
    .sort_values('atd_median')
    .round(2)
)
flow_summary['trip_pct'] = (flow_summary['trips'] / flow_summary['trips'].sum() * 100).round(1)
flow_summary['trips_volume_rank'] = flow_summary['trips'].rank(ascending=False).astype(int)

print('Courier Flow Summary (sorted by median ATD ascending = best to worst):')
display(flow_summary)

In [ ]:
# Geo archetype vs courier flow performance
geo_flow = (
    df.groupby(['geo_archetype', 'courier_flow'], observed=True)
    .agg(trips=(ATD_COL, 'count'), atd_median=(ATD_COL, 'median'))
    .reset_index()
)

pivot_geo_flow = geo_flow.pivot(index='geo_archetype', columns='courier_flow', values='atd_median').fillna(0)

fig, ax = plt.subplots(figsize=(14, max(4, len(pivot_geo_flow) * 0.5)))
sns.heatmap(
    pivot_geo_flow, ax=ax, cmap='YlOrRd',
    annot=True, fmt='.0f',
    cbar_kws={'label': 'Median ATD (min)'},
    linewidths=0.3,
)
ax.set_title('Median ATD — Geo Archetype × Courier Flow', fontweight='bold')
ax.set_xlabel('Courier Flow')
ax.set_ylabel('Geo Archetype')
plt.tight_layout()
plt.show()

---
## 7. Driver Incentive Targeting

We identify which driver segments to target with performance coaching vs positive incentives.

In [ ]:
driver_perf = (
    df[df['driver_uuid'].notna()]
    .groupby('driver_uuid')
    .agg(
        trips=(ATD_COL, 'count'),
        atd_median=(ATD_COL, 'median'),
        atd_std=(ATD_COL, 'std'),
        late_30_pct=(ATD_COL, lambda x: (x > 30).mean() * 100),
    )
    .query('trips >= 20')
    .reset_index()
)

driver_perf['atd_cv'] = driver_perf['atd_std'] / driver_perf['atd_median']

# Tier classification based on median ATD and late rate
med_atd      = driver_perf['atd_median'].median()
high_late_pct = driver_perf['late_30_pct'].quantile(0.75)

def tier(row):
    if row['atd_median'] <= med_atd * 0.9 and row['late_30_pct'] <= high_late_pct * 0.75:
        return 'Elite'
    if row['atd_median'] <= med_atd and row['late_30_pct'] <= high_late_pct:
        return 'Good'
    if row['atd_median'] > med_atd * 1.2 or row['late_30_pct'] > high_late_pct:
        return 'Needs Improvement'
    return 'Average'

driver_perf['tier'] = driver_perf.apply(tier, axis=1)

tier_summary = driver_perf.groupby('tier').agg(
    n_drivers=('driver_uuid', 'count'),
    total_trips=('trips', 'sum'),
    median_atd=('atd_median', 'median'),
    avg_late_pct=('late_30_pct', 'mean'),
).round(1)

tier_summary['pct_drivers'] = (tier_summary['n_drivers'] / tier_summary['n_drivers'].sum() * 100).round(1)
print('Driver Tier Distribution:')
display(tier_summary)

In [ ]:
TIER_COLORS = {
    'Elite': 'gold', 'Good': 'seagreen',
    'Average': 'steelblue', 'Needs Improvement': 'tomato'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatter: late rate vs median ATD
sample_d = driver_perf.sample(min(6000, len(driver_perf)), random_state=SEED)
for t, grp in sample_d.groupby('tier'):
    axes[0].scatter(grp['atd_median'], grp['late_30_pct'],
                    alpha=0.3, s=12, label=t, color=TIER_COLORS.get(t, 'gray'))
axes[0].axvline(med_atd, color='black', linestyle='--', linewidth=0.8)
axes[0].axhline(high_late_pct, color='black', linestyle='--', linewidth=0.8)
axes[0].set_xlabel('Driver Median ATD (min)')
axes[0].set_ylabel('% Late Trips (> 30 min)')
axes[0].set_title('Driver Performance Matrix')
axes[0].legend(title='Tier', fontsize=9)

# Trip volume by tier
tier_summary['total_trips'].sort_values(ascending=False).plot(
    kind='bar', ax=axes[1],
    color=[TIER_COLORS.get(t, 'gray') for t in tier_summary.sort_values('total_trips', ascending=False).index],
    edgecolor='none'
)
axes[1].set_title('Total Trip Volume by Driver Tier')
axes[1].set_ylabel('Total Trips')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Driver Incentive Targeting', fontweight='bold')
plt.tight_layout()
plt.show()

elite_trips = tier_summary.loc['Elite', 'total_trips'] if 'Elite' in tier_summary.index else 0
ni_trips    = tier_summary.loc['Needs Improvement', 'total_trips'] if 'Needs Improvement' in tier_summary.index else 0
print(f'\nElite drivers carry {elite_trips:,.0f} trips — protect their availability during peaks.')
print(f'Needs Improvement drivers carry {ni_trips:,.0f} trips — highest coaching ROI.')

---
## 8. Model Business Value Quantification

Translating MAE improvement into operational value.

In [ ]:
# Business assumptions
AVG_DAILY_TRIPS      = daily_avg_trips
BASELINE_MAE_ASSUMED = 12.0   # minutes — typical segment-median baseline
MODEL_MAE_ASSUMED    = 9.0    # minutes — LightGBM (adjust with actual from nb04)

# Load actual model results if available
import json
meta_path = Path('../model/model_metadata.json')
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    MODEL_MAE_ASSUMED = meta.get('test_mae', MODEL_MAE_ASSUMED)
    BASELINE_MAE_ASSUMED = BASELINE_MAE_ASSUMED
    print(f'Using actual model MAE from notebook 04: {MODEL_MAE_ASSUMED:.2f} min')

mae_improvement_min  = BASELINE_MAE_ASSUMED - MODEL_MAE_ASSUMED
mae_improvement_pct  = mae_improvement_min / BASELINE_MAE_ASSUMED * 100

# Conservative impact: 10% of MAE improvement translates to actual routing changes
REALIZATION_RATE = 0.10
LATE_ORDER_COST  = 2.5   # USD: estimated cost per late order (refund + voucher + churn risk)

# 1% improvement in on-time rate
trips_affected    = AVG_DAILY_TRIPS * (mae_improvement_min * REALIZATION_RATE / atd_median)
daily_cost_saving = trips_affected * LATE_ORDER_COST
annual_saving     = daily_cost_saving * 365

print('\n── Model Business Value Estimate ──────────────────────────')
print(f'  Avg daily trips          : {AVG_DAILY_TRIPS:,.0f}')
print(f'  Baseline MAE             : {BASELINE_MAE_ASSUMED:.1f} min')
print(f'  Model MAE                : {MODEL_MAE_ASSUMED:.1f} min')
print(f'  MAE improvement          : {mae_improvement_min:.1f} min ({mae_improvement_pct:.1f}%)')
print(f'  Realization rate (conservative): {REALIZATION_RATE*100:.0f}%')
print(f'  Trips improved/day       : {trips_affected:,.0f}')
print(f'  Estimated daily saving   : USD {daily_cost_saving:,.0f}')
print(f'  Estimated annual saving  : USD {annual_saving:,.0f}')
print('  (Assumptions: ${:.2f}/late order, {:.0f}% realization rate)'.format(
    LATE_ORDER_COST, REALIZATION_RATE * 100))

---
## 9. Strategic Recommendations

In [ ]:
recommendations = [
    {
        'Priority': '🔴 HIGH',
        'Area': 'Dispatch Optimization',
        'Finding': 'Pickup distance is the top controllable driver of ATD. '
                   'Trips with pickup > 1.5km are significantly slower.',
        'Action': 'Tighten the courier assignment radius during lunch (12–14h) '
                  'and dinner (19–23h) peaks. Set a hard cap of 1.5km pickup distance '
                  'or trigger pre-positioning 30 min before peak.',
        'KPI': 'Reduce avg pickup_distance by 15%',
    },
    {
        'Priority': '🔴 HIGH',
        'Area': 'Supply Pre-Positioning',
        'Finding': 'ATD spikes during dinner hours (19–23h local) despite being the '
                   'highest volume period — classic supply-demand mismatch.',
        'Action': 'Activate surge incentives 45 minutes before dinner peak to pre-position '
                  'couriers. Use predictive model output to forecast which territories '
                  'will need 20%+ more couriers than baseline.',
        'KPI': 'Reduce peak-hour median ATD by ≥ 3 min',
    },
    {
        'Priority': '🟡 MEDIUM',
        'Area': 'Driver Coaching Program',
        'Finding': 'Drivers in the Needs Improvement tier account for a meaningful share '
                   'of trips but have ATD significantly above the territory median.',
        'Action': 'Launch targeted coaching for bottom-quartile drivers using Z-score '
                  'ranking vs territory peers. Focus on route familiarity and pickup '
                  'efficiency. Track improvement over 4 weeks.',
        'KPI': 'Move 15% of Needs Improvement drivers to Average tier in 30 days',
    },
    {
        'Priority': '🟡 MEDIUM',
        'Area': 'Merchant Surface Optimization',
        'Finding': 'POS and Tablet surfaces show materially different ATD patterns, '
                   'suggesting varying restaurant preparation readiness times.',
        'Action': 'Investigate POS-connected restaurants for slower order acceptance. '
                  'Build a merchant readiness score to adjust courier dispatch timing '
                  '(delay courier send until restaurant confirms prep).',
        'KPI': 'Reduce eater_wait_min for POS surface by 2 min',
    },
    {
        'Priority': '🟢 STANDARD',
        'Area': 'Predictive ETA Integration',
        'Finding': 'LightGBM model achieves meaningful MAE improvement over segment baselines. '
                   'Model uses 22 features with distances and time as top predictors.',
        'Action': 'Deploy model via weekly ETL pipeline (ADF). Feed predictions into '
                  'Streamlit dashboard for ops teams. Re-train monthly with new data. '
                  'Use predictions to set dynamic customer-facing ETAs.',
        'KPI': 'MAE < 9 min on next month holdout test',
    },
    {
        'Priority': '🟢 STANDARD',
        'Area': 'Geo Archetype Strategy',
        'Finding': 'Drive momentum zones have distinct ATD patterns vs urban zones. '
                   'Non-standard geo archetypes may not have sufficient courier density.',
        'Action': 'Analyze order rejection rates by geo archetype to identify zones '
                  'where demand outstrips supply. Consider geo-targeted driver incentives.',
        'KPI': 'Reduce ATD variance by geo_archetype by 10%',
    },
]

rec_df = pd.DataFrame(recommendations)
print('Strategic Recommendations:')
for i, row in rec_df.iterrows():
    print(f'\n{row["Priority"]} — {row["Area"]}')
    print(f'  Finding : {row["Finding"]}')
    print(f'  Action  : {row["Action"]}')
    print(f'  KPI     : {row["KPI"]}')

---
## 10. Execution Roadmap

In [ ]:
roadmap = {
    'Week 1–2 (Foundation)': [
        'Deploy weekly ETL pipeline (ADF) — SQL query → AA_tables schema',
        'Launch Streamlit dashboard with EDA views and KPI cards',
        'Identify top 5 high-gap segments (territory × time block)',
    ],
    'Week 3–4 (Model Integration)': [
        'Integrate LightGBM model into weekly pipeline',
        'Add model prediction view to dashboard',
        'Set up monitoring: MAE alert if > 12 min on weekly holdout',
    ],
    'Month 2 (Operational Impact)': [
        'Pilot dispatch optimization: tighter pickup radius during peaks in 2 territories',
        'Launch Needs Improvement driver coaching cohort',
        'Implement merchant readiness alerting for POS surface',
    ],
    'Month 3+ (Scale & Iterate)': [
        'A/B test pre-positioning incentives during dinner peaks',
        'Expand model to other regions (Brazil, Colombia)',
        'Add weather, holiday, and event features to model',
        'Retrain model monthly; automate via ADF trigger',
    ],
}

print('═' * 65)
print('  EXECUTION ROADMAP')
print('═' * 65)
for phase, items in roadmap.items():
    print(f'\n  ▶ {phase}')
    for item in items:
        print(f'    ✓ {item}')
print('\n' + '═' * 65)

In [ ]:
# Final summary visualization: opportunity matrix
opportunities = {
    'Dispatch\nOptimization':    {'impact': 9, 'effort': 4, 'size': 800},
    'Supply\nPre-positioning':   {'impact': 8, 'effort': 5, 'size': 600},
    'Driver\nCoaching':          {'impact': 6, 'effort': 3, 'size': 400},
    'Merchant\nReadiness':       {'impact': 5, 'effort': 5, 'size': 300},
    'Predictive\nETA Model':     {'impact': 7, 'effort': 7, 'size': 700},
    'Geo Archetype\nStrategy':   {'impact': 4, 'effort': 6, 'size': 250},
    'Dashboard\nDeployment':     {'impact': 6, 'effort': 2, 'size': 350},
    'Model\nMonitoring':         {'impact': 5, 'effort': 3, 'size': 250},
}

fig, ax = plt.subplots(figsize=(12, 7))

for name, vals in opportunities.items():
    ax.scatter(vals['effort'], vals['impact'],
               s=vals['size'], alpha=0.6,
               color='tomato' if vals['impact'] >= 7 else ('goldenrod' if vals['impact'] >= 5 else 'steelblue'),
               edgecolors='black', linewidths=0.5)
    ax.annotate(name, (vals['effort'], vals['impact']),
                ha='center', va='center', fontsize=8, fontweight='bold')

ax.axvline(5, color='gray', linestyle='--', alpha=0.5)
ax.axhline(5.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Implementation Effort (1=Easy, 10=Hard)', fontsize=11)
ax.set_ylabel('Business Impact (1=Low, 10=High)', fontsize=11)
ax.set_title('Opportunity Matrix: Impact vs Effort', fontweight='bold', fontsize=13)
ax.set_xlim(0, 11)
ax.set_ylim(0, 11)

# Quadrant labels
ax.text(1.5, 10.2, 'Quick Wins', fontsize=9, color='seagreen', fontweight='bold')
ax.text(7.5, 10.2, 'Major Projects', fontsize=9, color='tomato', fontweight='bold')
ax.text(1.5, 0.5, 'Fill-ins', fontsize=9, color='steelblue')
ax.text(7.5, 0.5, 'Questionable', fontsize=9, color='gray')

plt.tight_layout()
plt.show()